# Name That LLM - Zephyr-7b Quantized LoRA
Adapted for the 18-class classification task with `.parquet` submission format.

In [ ]:
!pip install -U transformers accelerate peft bitsandbytes datasets pandas numpy scikit-learn -q

In [ ]:
!nvidia-smi

In [ ]:
TARGET_MODEL = "HuggingFaceH4/zephyr-7b-alpha"
DEBUG = False

# The exact 18 model names from the competition rules
MODEL_NAMES = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]
NUM_LABELS = len(MODEL_NAMES)

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    BitsAndBytesConfig, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.model_selection import GroupKFold
import pandas as pd
import numpy as np
import torch
import os

In [ ]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

In [ ]:
# Adjust path if necessary, e.g., '/kaggle/input/competitions/name-that-llm/'
DATA_DIR = "/kaggle/input/competitions/name-that-llm/" 

try:
    train = pd.read_csv(f"{DATA_DIR}train.csv")
    test = pd.read_csv(f"{DATA_DIR}test.csv")
except FileNotFoundError:
    train = pd.read_csv("train.csv")
    test = pd.read_csv("test.csv")

# Dynamically find the text column (could be 'text', 'passage', or 'response')
text_col = 'text' if 'text' in train.columns else ('passage' if 'passage' in train.columns else train.columns[1])
train['text'] = train[text_col].fillna('')
test['text'] = test[text_col].fillna('')

# Map string labels to integer indices for training
train['label'] = train['model'].apply(lambda x: MODEL_NAMES.index(x) if x in MODEL_NAMES else 0)

# GroupKFold requires a group column to prevent data leakage. Use 'prompt' if it exists, otherwise 'id'
group_col = 'prompt' if 'prompt' in train.columns else 'id'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples, max_length=512):
    return tokenizer(examples["text"], truncation=True, max_length=max_length, padding=False)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
folds = GroupKFold(n_splits=5)
train['fold'] = -1

for i, (_, test_index) in enumerate(folds.split(train, train['label'], groups=train[group_col])): 
    train.loc[test_index, 'fold'] = i

final_test_preds = []

for fold in range(5):
    print(f'\n{"="*50}\nFold {fold+1}/5\n{"="*50}')
    
    valid_df = train[train["fold"] == fold].copy()
    train_df = train[train["fold"] != fold].copy()
    
    train_ds = Dataset.from_pandas(train_df)
    valid_ds = Dataset.from_pandas(valid_df)
    
    train_tokenized_ds = train_ds.map(preprocess_function, batched=True)
    valid_tokenized_ds = valid_ds.map(preprocess_function, batched=True)
    
    # Changed to AutoModelForSequenceClassification to prevent Mistral/Zephyr architecture mismatch warnings
    base_model = AutoModelForSequenceClassification.from_pretrained(
        TARGET_MODEL,
        num_labels=NUM_LABELS,
        quantization_config=bnb_config,
        device_map={"": 0}
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id
    
    model = get_peft_model(base_model, peft_config)
    
    def print_trainable_parameters(model):
        trainable_params = 0
        all_param = 0
        for _, param in model.named_parameters():
            all_param += param.numel()
            if param.requires_grad:
                trainable_params += param.numel()
        print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")
    
    print_trainable_parameters(model)
    
    steps = 50 if DEBUG else 200
    
    training_args = TrainingArguments(
        output_dir=f"zephyr_fold{fold}",
        learning_rate=1e-4,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        max_grad_norm=0.3,
        optim='paged_adamw_32bit',
        lr_scheduler_type="cosine",
        num_train_epochs=2,
        weight_decay=0.01,
        evaluation_strategy="steps",
        save_strategy="steps",
        save_steps=steps,
        eval_steps=steps,
        logging_steps=steps,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        push_to_hub=False,
        warmup_ratio=0.03,
        report_to='none',
        bf16=True
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized_ds,
        eval_dataset=valid_tokenized_ds,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    
    trainer.train()
    
    save_dir = f"zephyr_fold{fold}"
    trainer.save_model(save_dir)
    
    test_ds = Dataset.from_pandas(test)
    test_tokenized_ds = test_ds.map(preprocess_function, batched=True)
    
    pred_output = trainer.predict(test_tokenized_ds)
    logits = pred_output.predictions
    
    # Softmax to get probabilities
    e_x = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probas = e_x / e_x.sum(axis=1, keepdims=True)
    final_test_preds.append(probas)
    
    del trainer, model, base_model
    torch.cuda.empty_cache()

In [ ]:
# Average predictions across all 5 folds
avg_probas = np.mean(final_test_preds, axis=0)

# Get the predicted class index for each row
predicted_indices = np.argmax(avg_probas, axis=1)

# Map indices back to the exact string model names required by the competition
predicted_models = [MODEL_NAMES[i] for i in predicted_indices]

# Create submission dataframe with exactly 'id' and 'model' columns
submission = pd.DataFrame({
    'id': test['id'],
    'model': predicted_models
})

# Save as parquet (REQUIRED by competition rules)
submission.to_parquet('submission.parquet', index=False)
print("Submission saved to submission.parquet")
print(submission.head())